# Entropy baseline — отбор данных по энтропии base-модели

Эксперимент: отбираем 90 000 примеров с **наивысшей средней token-level энтропией** распределения базовой `Qwen2.5-1.5B` на assistant-токенах. Сравниваем с `baseline_masked_packed_base_90k_1ep_noleak` (random 90k, 1 эпоха) на той же `assistant_only_PPL`.

Получен из `selection_template.ipynb` через `cp` + правки в Cell 10 (`STRATEGY` + параметры скоринга) и Cell 12 (entropy-логика в STRATEGY BLOCK), плюс ячейка сравнения с baseline. Этот ноутбук теперь используется как обновлённый selection-шаблон для следующих стратегий: из него нужно тянуть стек base-модели, bf16-only, `90k × 1`, `warmup_ratio`, fixed ChatML и common-val comparison.

## Режимы запуска

- **Fast mode (по умолчанию):** `POOL_LIMIT = 200_000`, `STRATEGY = "entropy_90k_from_200k_noleak"`. Скоринг считается по random-200k пулу, затем берётся top-90k.
- **Full mode:** переключи `POOL_LIMIT = None` и `STRATEGY = "entropy_90k"`. Скоринг идёт по всему датасету.

Оба варианта валидны методологически; fast mode фиксирует параметр эксперимента «entropy на random-200k подвыборке» и заявляется как таковой.

## Что унаследовано от `baseline_masked_packed.ipynb`

- Обучение: `Qwen2.5-1.5B` + QLoRA (r=16) + masked loss через `train_on_responses_only` + `packing=True`.
- Precision: только `bf16`; если GPU не поддерживает bf16, ноутбук останавливается. Fallback на fp16 намеренно отключён.
- Данные: `SUBSAMPLE_SIZE = 90_000`, `NUM_EPOCHS = 1`. Это заменяет старый режим `30k × 3`: compute близкий, но уникальных примеров в 3 раза больше и ниже риск переобучения на маленькой подвыборке.
- Conversational датасет рендерится в `text` через явно зафиксированный Qwen2.5 ChatML-шаблон `get_chat_template(..., "qwen-2.5")` до создания тренера.
- `train_on_responses_only` после создания SFTTrainer корректно обрабатывает множественные пары маркеров `<|im_start|>user\n` / `<|im_start|>assistant\n` в одной упакованной последовательности (Unsloth discussion #2828).

## Метрика

**Метрика для сравнения стратегий — `assistant-only PPL`, а не full-text PPL.** Full-text PPL остаётся диагностикой, но verdict строится только на внешнем common val: `shuffle(seed=SEED)[0:COMMON_VAL_HOLDOUT_SIZE]`. Все train/selection/scoring pools начинаются после этого holdout, поэтому финальная метрика не смешивается с обучающими данными.


## Шаг 1.1 — Установка зависимостей

Чистая переустановка ключевых пакетов + установка свежего `unsloth` с PyPI (он сам подберёт совместимые `transformers`, `datasets`, `trl`, `peft`, `torchao`). Жёстко пинить версии не пытаемся: сборки Colab меняются, Unsloth выкатывает новый релиз каждый месяц, и попытка зафиксировать всё ломается на следующей неделе.

После установки **обязательно** перезапустить runtime (Colab поднимет уведомление в конце ячейки). После рестарта продолжаем со следующей ячейки.

In [ ]:
# Сносим всё, что мы могли поставить кривой версии в предыдущих попытках.
# Ошибки игнорируем — пакета может и не быть.
!pip uninstall -y -q \
    transformers datasets trl peft \
    unsloth unsloth_zoo torchao \
    2>/dev/null

# Ставим unsloth с PyPI — он сам подтянет совместимые transformers/datasets/trl/peft/torchao.
!pip install --quiet --upgrade unsloth

# Дополняем тем, что unsloth не тянет, но нам нужно для SFT и квантизации.
!pip install --quiet --upgrade bitsandbytes accelerate sentencepiece

## Шаг 1.1.5 — Проверка версий

Печатаем фактические версии установленных пакетов. От этого зависит API SFTConfig (в `trl>=0.20` `max_seq_length` переименован в `max_length`, а `dataset_text_field` депрекейтнут), а также то, какие функции Unsloth доступны.

Запускать **после** рестарта runtime, прежде чем идти дальше.

In [ ]:
import importlib.metadata as _md

print("Установленные версии:")
for pkg in [
    "unsloth", "unsloth_zoo",
    "transformers", "trl", "peft", "datasets",
    "torchao", "bitsandbytes", "accelerate",
    "torch",
]:
    try:
        v = _md.version(pkg)
    except _md.PackageNotFoundError:
        v = "MISSING"
    print(f"  {pkg:15s} {v}")

## Шаг 1.2 — Проверка железа и обязательный bf16

T4 (Turing, sm_75) **не поддерживает bf16**. A100/L4 (Ampere+, sm_80+) — поддерживает. Если bf16 недоступен, ноутбук должен остановиться: fallback на fp16 в этом эксперименте запрещён.

Не используем `torch.cuda.is_bf16_supported()`: в PyTorch есть [баг](https://github.com/pytorch/pytorch/issues/118122), из-за которого функция возвращает `True` для Turing. Проверяем напрямую через compute capability.

> **Важно:** `import unsloth` обязательно делаем здесь, до любого импорта из `transformers`. Иначе теряются оптимизации Unsloth (FastAttention, патчи torch и т.д.) и появляется warning про порядок импортов.


In [ ]:
import unsloth   # ОБЯЗАТЕЛЬНО до transformers
import torch


def detect_precision():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна. Для этого запуска нужен GPU с bf16 (A100/L4/H100 или другой sm_80+).")
    major, minor = torch.cuda.get_device_capability(0)
    if major < 8:
        raise RuntimeError(
            f"bf16 недоступен на GPU sm_{major}{minor}. "
            "Fallback на fp16 отключён намеренно; запусти на A100/L4/H100 или другом sm_80+."
        )
    return {"bf16": True, "fp16": False}


def check_hardware():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна. Для этого запуска нужен GPU с bf16 (A100/L4/H100 или другой sm_80+).")

    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    major, minor = torch.cuda.get_device_capability(0)
    p = detect_precision()

    print(f"GPU:                {name}")
    print(f"VRAM:               {vram_gb:.1f} GB")
    print(f"Compute capability: sm_{major}{minor}")
    print("Precision:          bf16 (required)")

    if vram_gb < 4:
        print("СОВЕТ: VRAM < 4 GB — переключитесь на Qwen2.5-0.5B")
    elif vram_gb < 8:
        print("OK: Qwen2.5-1.5B + 4bit поместится с запасом")
    else:
        print("OK: можно использовать Qwen2.5-1.5B или 3B")


check_hardware()


## Шаг 1.2.5 — Google Drive (для устойчивости к падению сессии)

Чекпоинты копятся каждые 200 шагов. Если сессия Colab упадёт (а она упадёт рано или поздно — таймаут, разрыв сети, OOM), всё в `/content/` пропадает. Чтобы не терять часы обучения, монтируем Google Drive и сохраняем туда.

После монтирования: даже после полного дисконнекта можно заново открыть ноутбук, прогнать все ячейки сверху вниз — `trainer.train(resume_from_checkpoint=True)` подхватит с последнего чекпоинта.

Если запускаешь локально (не Colab) или не хочешь Drive — поставь `USE_DRIVE = False`, сохранение пойдёт в текущую папку.

Для параллельных запусков: если `/content/drive` уже занят локальными файлами или другой тетрадкой, ячейка пробует запасные mountpoint-ы (`/content/gdrive`, `/content/drive_1`, ...). `force_remount=True` намеренно не используется, чтобы не сломать соседний прогон.


In [ ]:
USE_DRIVE = True   # False = сохранять локально (потеряется при разрыве сессии)

if USE_DRIVE:
    try:
        import os
        from google.colab import drive

        def _is_drive_mounted(path):
            return os.path.isdir(os.path.join(path, "MyDrive"))

        def _mount_drive_parallel_safe():
            # Не используем force_remount=True: это может отвалить Drive у другой тетрадки.
            candidates = ["/content/drive", "/content/gdrive"] + [
                f"/content/drive_{i}" for i in range(1, 8)
            ]

            for mountpoint in candidates:
                if _is_drive_mounted(mountpoint):
                    print(f"Google Drive уже доступен: {mountpoint}")
                    return mountpoint

            last_error = None
            for mountpoint in candidates:
                try:
                    if os.path.exists(mountpoint):
                        if not os.path.isdir(mountpoint):
                            print(f"{mountpoint} занят файлом — пробуем следующий")
                            continue
                        if os.listdir(mountpoint):
                            print(f"{mountpoint} занят локальными файлами — пробуем следующий")
                            continue

                    os.makedirs(mountpoint, exist_ok=True)
                    drive.mount(mountpoint)
                    if _is_drive_mounted(mountpoint):
                        return mountpoint
                    print(f"WARNING: {mountpoint} смонтирован, но MyDrive не найден; используем как есть")
                    return mountpoint
                except ValueError as exc:
                    last_error = exc
                    if _is_drive_mounted(mountpoint):
                        return mountpoint
                    print(f"{mountpoint}: {exc}")

            raise RuntimeError(
                "Не удалось подключить Google Drive: все mountpoint заняты. "
                "Очисти локальные /content/drive* или поставь USE_DRIVE=False."
            ) from last_error

        DRIVE_MOUNT = _mount_drive_parallel_safe()
        OUTPUT_BASE = f"{DRIVE_MOUNT}/MyDrive/diploma"
        print(f"OUTPUT_BASE = {OUTPUT_BASE} (Google Drive)")
    except ImportError:
        print("google.colab недоступен — сохраняем локально")
        OUTPUT_BASE = "."
else:
    OUTPUT_BASE = "."
    print(f"OUTPUT_BASE = {OUTPUT_BASE} (локально)")


## Шаг 1.3 — Конфиг

Все константы в одном месте. Между экспериментами **меняется только `STRATEGY`** — `OUTPUT_DIR` собирается из неё автоматически, поэтому артефакты разных запусков не пересекаются по папкам.

`SUBSAMPLE_SIZE = 90_000`, `NUM_EPOCHS = 1` — обновлённый бюджет selection-экспериментов. Он соответствует новому baseline-протоколу: вместо `30_000 × 3 эпохи` модель один раз проходит по 90k уникальных примеров.

Все остальные гиперпараметры (lr, batch, seq-len, LoRA rank, ChatML render, common val) **должны совпадать** между экспериментами — иначе сравнение нечестное.


In [ ]:
# =============================================================================
# ★ ИМЯ СТРАТЕГИИ — это единственное, что обычно меняется между экспериментами.
# =============================================================================
# Используется в:
#   1) OUTPUT_DIR — папка с артефактами этого эксперимента;
#   2) HF_REPO_ID (опционально) — имя репозитория на Hub.
#
# Допустимые имена для дипломной таблицы после перехода на 90k × 1:
#   "random_90000"             — равномерный random (baseline_masked_packed_base_90k_1ep_noleak)
#   "entropy_90k_from_200k_noleak" — top-90k по энтропии из random-200k пула после common-val holdout
#   "loss_based_90k_from_200k" — top-90k по reducible loss из random-200k пула
#   "ifd_90k_from_200k"        — top-90k по IFD из random-200k пула
STRATEGY = "entropy_90k_from_200k_noleak"   # top-K по средней token-level энтропии base-модели на assistant-токенах
# =============================================================================

MODEL_NAME      = "Qwen/Qwen2.5-1.5B"
DATASET_NAME    = "d0rj/ru-instruct"

SUBSAMPLE_SIZE  = 90_000

MAX_SEQ_LEN     = 2048
BATCH_SIZE      = 4
GRAD_ACCUM      = 2                 # эффективный batch = BATCH_SIZE * GRAD_ACCUM = 8
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 1
WARMUP_RATIO    = 0.03
SEED            = 42
VAL_SIZE        = 0.05
COMMON_VAL_HOLDOUT_SIZE = int(SUBSAMPLE_SIZE * VAL_SIZE)  # внешний common val, исключён из train/selection pools

OUTPUT_DIR      = f"{OUTPUT_BASE}/outputs/{STRATEGY}"
ADAPTER_DIR     = f"{OUTPUT_DIR}/adapter"
LOG_PATH        = f"{OUTPUT_DIR}/training_log.json"
METRICS_PATH    = f"{OUTPUT_DIR}/final_metrics.json"
SCORES_PATH     = f"{OUTPUT_DIR}/scores.npy"     # кеш entropy-скоров
INDICES_PATH    = f"{OUTPUT_DIR}/selected_indices.npy"  # отобранные индексы (для воспроизводимости)

# --- Параметры entropy-скоринга (используются в Cell 12) ---
# POOL_LIMIT: размер пула, на котором считаем энтропии и из которого берём top-K.
#   None      = весь датасет (~753k, полный режим)
#   200_000   = fast mode, рекомендуется для дипломного запуска
# Отбор top-K всегда происходит ИЗ POOL_LIMIT (т.е. fast-mode дает entropy_90k_from_200k).
POOL_LIMIT             = 200_000
SCORE_BATCH_SIZE       = 8           # forward-only, 4-bit; на L4 стабильно до 16
SCORE_CHECKPOINT_EVERY = 10_000      # сохранять промежуточные скоры каждые N примеров

EVAL_SAMPLES        = 500
FINAL_EVAL_SAMPLES  = 200

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

if POOL_LIMIT is not None and SUBSAMPLE_SIZE > POOL_LIMIT:
    raise ValueError(f"SUBSAMPLE_SIZE={SUBSAMPLE_SIZE} больше POOL_LIMIT={POOL_LIMIT}")

# Sanity-check: не позволяем случайно запустить шаблон под именем-плейсхолдером.
if STRATEGY.startswith("TEMPLATE_"):
    print(f"⚠️  STRATEGY = {STRATEGY!r} — это плейсхолдер из шаблона.")
    print(f"   Перепиши STRATEGY в Cell 10 под имя своего эксперимента.")
    print(f"   Сейчас будет создан OUTPUT_DIR = {OUTPUT_DIR}")
print(f"\nSTRATEGY:    {STRATEGY}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")


## Шаг 1.3.5 — Опционально: удалить артефакты прошлого прогона

В следующей ячейке **`RESET_PREVIOUS_RUN = True`** полностью удаляет **`OUTPUT_DIR`** стратегии (чекпоинты, `adapter/`, скоры, `il_*` для RHO — всё внутри этой папки). Нужен для **чистого** перезапуска без resume и без смешивания метрик.

**Внимание:** на Google Drive удаление необратимо.

> Нумерация ячеек ниже по файлу сдвинута на **+2** относительно старых упоминаний «Cell 12 / 16 / 24» в markdown — имей в виду при поиске по тексту ноутбука.


In [ ]:
import os
import shutil

RESET_PREVIOUS_RUN = False  # True = полностью снести OUTPUT_DIR и создать заново

if RESET_PREVIOUS_RUN:
    if os.path.isdir(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
        print(f"Удалено: {OUTPUT_DIR}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("Пустая OUTPUT_DIR создана.")
else:
    print("RESET_PREVIOUS_RUN=False — OUTPUT_DIR не трогаем.")


## Шаг 1.4 — Подготовка датасета + ★ entropy-стратегия отбора

Структура ячейки:

1. **Загрузка полного датасета** (`ds_full`).

2. **Утилиты entropy-скоринга** — multi-turn-safe функции:
   - `_find_assistant_positions` — поиск позиций в logits, где модель предсказывает assistant-токен (через маркеры `<|im_start|>assistant\n` / `<|im_start|>user\n` напрямую в `input_ids`).
   - `compute_assistant_entropies_batch` — батчированный forward с `float()`-upcast перед softmax.
   - `load_partial_scores` + `score_dataset(ds_full)` — resumable скоринг с чекпоинтами каждые `SCORE_CHECKPOINT_EVERY` примеров и length-bucketing pre-pass.

3. **★ STRATEGY BLOCK** — entropy top-K:
   - Загружает `scorer_model` (та же `Qwen2.5-1.5B`, **4-bit, без обучения**).
   - Явно фиксирует `get_chat_template(..., chat_template="qwen-2.5")` для scorer/tokenizer.
   - Вызывает `score_dataset(ds_full)` → возвращает `scores` и `pool_ds`.
   - `pool_ds` строится из `ds_full.shuffle(seed=SEED)` **после** внешнего common-val holdout: fast-mode берёт `[COMMON_VAL_HOLDOUT_SIZE : COMMON_VAL_HOLDOUT_SIZE + POOL_LIMIT]`, full-mode — всё после holdout.
   - Top-K по убыванию энтропии: `selected_indices = np.argsort(-valid_scores)[:SUBSAMPLE_SIZE]` (NaN отфильтрованы). Common-val примеры недоступны для отбора по построению.
   - **Освобождает VRAM** через `del scorer_model + gc + torch.cuda.empty_cache()` — иначе следующая ячейка не сможет загрузить обучающую копию.

4. **Применение ChatML template + train/val split** — `pool_ds.select(selected_indices)` (важно: не `ds_full`, потому что в fast-mode индексы живут в координатах pool_ds).

**Resume:** скоры кешируются в `SCORES_PATH = outputs/<STRATEGY>/scores.npy`. Если сессия упала — перезапуск Cell 12 продолжит с того места, где остановился. `INDICES_PATH` сохраняет финальный отбор для воспроизводимости.

**Sanity-чекпоинты:**
- Распределение энтропий (min/max/mean/std + квантили) — длинный правый хвост норма.
- NaN-rate (`% невалидных скоров`) — ожидаем < 5%; > 5% означает проблемы с маркерами или агрессивную обрезку.
- Top-K mean entropy должно быть существенно больше pool mean (иначе отбор не работает).
- Стандартные length-stats (`avg_len_tokens`, `pct_over_max_seq_len`).


In [ ]:
import json
import numpy as np
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer


# ----- ШАГ 1: загрузка полного датасета (без отбора) -----
ds_full = load_dataset(DATASET_NAME, split="train")
print(f"Полный размер ds_full: {len(ds_full)} примеров")

tok = get_chat_template(
    AutoTokenizer.from_pretrained(MODEL_NAME),
    chat_template="qwen-2.5",
)


# =============================================================================
# Утилиты для entropy-скоринга (используются в STRATEGY BLOCK ниже).
# Multi-turn-safe: ищем маркеры <|im_start|>user\n / <|im_start|>assistant\n
# напрямую в input_ids — та же логика, что в train_on_responses_only и в
# compute_assistant_only_perplexity из baseline_masked_packed.
# =============================================================================
import torch
import torch.nn.functional as F


def _find_assistant_positions(ids, asst_marker_ids, user_marker_ids, valid_len):
    """Позиции i в logits, где предсказывается assistant-токен.

    - Идём по input_ids, ищем вхождения <|im_start|>assistant\\n.
    - Конец assistant-сегмента = следующий <|im_start|>user\\n или конец
      валидной части (без padding).
    - Сдвиг на 1: токен на позиции k предсказывается logits[k-1].
    """
    positions = set()
    i = 0
    while i < valid_len:
        if ids[i:i + len(asst_marker_ids)] == asst_marker_ids:
            start = i + len(asst_marker_ids)
            end = valid_len
            for j in range(start, valid_len - len(user_marker_ids) + 1):
                if ids[j:j + len(user_marker_ids)] == user_marker_ids:
                    end = j
                    break
            for k in range(start, end):
                if 0 < k <= valid_len - 1:
                    positions.add(k - 1)
            i = end
        else:
            i += 1
    return positions


@torch.no_grad()
def compute_assistant_entropies_batch(messages_batch, model, tok, max_len=MAX_SEQ_LEN):
    """Средняя token-level энтропия (в натах) на assistant-токенах для батча.

    Возвращает list[float] длины len(messages_batch).
    NaN если у примера нет валидных assistant-позиций (всё обрезалось при truncation).
    """
    # _find_assistant_positions сканит ids[0..valid_len), что корректно только при
    # right-padding. При left-padding реальные данные лежат в [L - valid_len, L),
    # и поиск маркеров не найдёт ничего, кроме самого длинного примера в батче
    # Для скоринга нужен right-padding: поиск assistant-маркеров идёт по валидному префиксу.
    assert tok.padding_side == "right", (
        f"padding_side='{tok.padding_side}' — установи tok.padding_side='right' "
        f"после загрузки токенайзера, иначе скоринг сломается."
    )
    user_marker_ids = tok("<|im_start|>user\n",      add_special_tokens=False).input_ids
    asst_marker_ids = tok("<|im_start|>assistant\n", add_special_tokens=False).input_ids

    texts = [
        tok.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in messages_batch
    ]
    enc = tok(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_len,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}

    logits = model(**enc).logits                            # [B, L, V]
    log_probs = F.log_softmax(logits.float(), dim=-1)       # float() — fp16 нестабилен на vocab~150k
    probs     = log_probs.exp()
    entropies = -(probs * log_probs).sum(dim=-1)            # [B, L], в натах

    results = []
    for b in range(len(messages_batch)):
        ids = enc["input_ids"][b].tolist()
        valid_len = int(enc["attention_mask"][b].sum().item())

        asst_positions = _find_assistant_positions(
            ids, asst_marker_ids, user_marker_ids, valid_len,
        )
        if not asst_positions:
            results.append(float("nan"))
            continue

        positions = torch.tensor(sorted(asst_positions), device=entropies.device)
        results.append(entropies[b, positions].mean().item())

    return results


# =============================================================================
# Скоринг датасета: resumable + чекпоинты + length-bucketing pre-pass.
# Использует compute_assistant_entropies_batch выше и глобальные scorer_model /
# scorer_tok (грузятся в STRATEGY BLOCK прямо перед score_dataset()).
# =============================================================================
import os
import gc
from tqdm.auto import tqdm


def load_partial_scores(path, n_total):
    """Загружает уже посчитанные скоры (для resume) или создаёт NaN-массив."""
    if os.path.exists(path):
        scores = np.load(path)
        if len(scores) != n_total:
            raise RuntimeError(
                f"Размер кеша {path} ({len(scores)}) не совпадает с пулом ({n_total}). "
                f"Удали файл, если хочешь пересчитать на новом пуле."
            )
        n_done = int((~np.isnan(scores)).sum())
        print(f"Найдено {n_done}/{n_total} уже посчитанных скоров в {path}")
        return scores
    return np.full(n_total, np.nan, dtype=np.float32)


def score_dataset(ds_full):
    """Считает среднюю энтропию assistant-токенов для каждого примера в пуле.

    Resumable: при повторном запуске продолжает с того места, где упала сессия.
    Length-bucketed: сортирует todo-индексы по char-длине → соседние батчи имеют
    близкие длины → меньше padding waste.

    Возвращает (scores, pool_ds), где scores[i] соответствует pool_ds[i].
    pool_ds строится после внешнего common-val holdout; fast-mode = shuffle(seed=SEED)[COMMON_VAL_HOLDOUT_SIZE:COMMON_VAL_HOLDOUT_SIZE+POOL_LIMIT].
    """
    ds_shuffled = ds_full.shuffle(seed=SEED)
    if POOL_LIMIT is not None:
        pool_start = COMMON_VAL_HOLDOUT_SIZE
        pool_end = pool_start + POOL_LIMIT
        if pool_end > len(ds_shuffled):
            raise ValueError(
                f"COMMON_VAL_HOLDOUT_SIZE + POOL_LIMIT = {pool_end} больше размера датасета ({len(ds_shuffled)})"
            )
        pool_ds = ds_shuffled.select(range(pool_start, pool_end))
        print(
            f"Fast mode: пул {POOL_LIMIT} примеров из shuffle(seed={SEED})[{pool_start}:{pool_end}], "
            f"common val = [0:{COMMON_VAL_HOLDOUT_SIZE})"
        )
    else:
        pool_ds = ds_shuffled.select(range(COMMON_VAL_HOLDOUT_SIZE, len(ds_shuffled)))
        print(f"Full mode: пул = весь датасет после common-val holdout [0:{COMMON_VAL_HOLDOUT_SIZE})")
    print(f"Пул для скоринга: {len(pool_ds)} примеров")

    scores = load_partial_scores(SCORES_PATH, len(pool_ds))
    todo = np.where(np.isnan(scores))[0].tolist()
    if not todo:
        print("Все скоры уже посчитаны, скоринг пропускаем.")
        return scores, pool_ds

    # Length-bucketing pre-pass: ~5 мин на 750k, ~1.5 мин на 200k.
    print(f"Length-bucketing pre-pass для {len(todo)} примеров...")
    char_lens = []
    for i in tqdm(todo, desc="char lens"):
        msgs = pool_ds[i]["conversations"]
        char_lens.append(sum(len(m["content"]) for m in msgs))
    todo = [i for _, i in sorted(zip(char_lens, todo))]

    pbar = tqdm(total=len(todo), desc=f"Scoring (batch={SCORE_BATCH_SIZE})")
    since_save = 0

    for batch_start in range(0, len(todo), SCORE_BATCH_SIZE):
        batch_idx = todo[batch_start:batch_start + SCORE_BATCH_SIZE]
        messages_batch = [pool_ds[i]["conversations"] for i in batch_idx]

        try:
            ents = compute_assistant_entropies_batch(
                messages_batch, scorer_model, scorer_tok,
            )
        except Exception as e:
            print(f"\n[!] Батч idx={batch_idx[0]}..{batch_idx[-1]}: "
                  f"{type(e).__name__}: {e}")
            ents = [float("nan")] * len(batch_idx)
            # После OOM в CUDA-кеше остаётся фрагментация — без явной очистки
            # следующие (даже исправные) батчи начинают каскадно падать.
            torch.cuda.empty_cache()
            gc.collect()

        for i, ent in zip(batch_idx, ents):
            scores[i] = ent

        pbar.update(len(batch_idx))
        since_save += len(batch_idx)

        if since_save >= SCORE_CHECKPOINT_EVERY:
            np.save(SCORES_PATH, scores)
            since_save = 0

    pbar.close()
    np.save(SCORES_PATH, scores)
    nan_count = int(np.isnan(scores).sum())
    print(f"Готово. NaN в скорах: {nan_count}/{len(scores)} "
          f"({nan_count / len(scores) * 100:.2f}%)")
    return scores, pool_ds


# =============================================================================
# ★ ШАГ 2: СТРАТЕГИЯ ОТБОРА ДАННЫХ — ЕДИНСТВЕННОЕ МЕСТО ДЛЯ РЕДАКТИРОВАНИЯ ★
# =============================================================================
#
# Контракт:
#   - На вход: ds_full (Dataset с колонкой `conversations`).
#   - На выход: np.ndarray длины SUBSAMPLE_SIZE с индексами в pool_ds.
#
# В этом ноутбуке ниже реализован entropy top-K. Для других стратегий
# перепиши блок между «BEGIN» и «END», сохранив контракт.
#
# Подсказки для других стратегий:
#
# Entropy-based:
#   1) Загрузи базовую Qwen2.5 без LoRA.
#   2) Для каждого ex в pool_ds посчитай средний log-softmax entropy на
#      assistant-токенах (см. compute_assistant_only_perplexity для логики).
#   3) Кешируй scores в SCORES_PATH (np.save).
#   4) selected_indices = np.argsort(scores)[-SUBSAMPLE_SIZE:]  # top-K
#
# Loss-based:
#   1) Аналогично, но скор = средний CE-loss на assistant-токенах.
#   2) selected_indices = np.argsort(scores)[-SUBSAMPLE_SIZE:]  # высокий loss = сложные примеры
#
# Learned scoring:
#   1) Сначала отдельным ноутбуком обучи scoring-модель (например, регрессию
#      на embedding-ах базовой модели) на небольшом ручном labelset.
#   2) Прогони scoring-модель по ds_full, получишь scores.
#   3) selected_indices = np.argsort(scores)[-SUBSAMPLE_SIZE:]
#
# Во всех случаях — сохраняй scores в SCORES_PATH, чтобы при перезапуске
# не пересчитывать.
# =============================================================================
print(f"\n★ STRATEGY: {STRATEGY}")

# ---- BEGIN STRATEGY BLOCK ----
# 1) Загрузка scorer-модели (та же база Qwen2.5-1.5B, без обучения, 4-bit).
from unsloth import FastLanguageModel
print("Загружаем скорер-модель (4-bit, без обучения)...")
scorer_model, scorer_tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
scorer_tok = get_chat_template(scorer_tok, chat_template="qwen-2.5")
FastLanguageModel.for_inference(scorer_model)

# Для скоринга нужен right-padding: иначе _find_assistant_positions упрётся
# в pad-токены и вернёт NaN почти для всего батча, кроме самого длинного примера.
scorer_tok.padding_side = "right"
print(f"scorer_tok.padding_side = {scorer_tok.padding_side!r}")

# 2) Скоринг (resumable, кеш в SCORES_PATH).
scores, pool_ds = score_dataset(ds_full)

# 3) Распределение энтропий (для отчёта и sanity-check).
valid_mask    = ~np.isnan(scores)
valid_indices = np.where(valid_mask)[0]
valid_scores  = scores[valid_indices]
if len(valid_indices) < SUBSAMPLE_SIZE:
    raise RuntimeError(
        f"Валидных скоров {len(valid_indices)} < SUBSAMPLE_SIZE={SUBSAMPLE_SIZE}. "
        f"Снизь NaN-rate (проверь маркеры/truncation) или уменьши SUBSAMPLE_SIZE."
    )
print(f"\nРаспределение энтропий по пулу ({len(valid_scores)} валидных):")
print(f"  min={valid_scores.min():.3f}  max={valid_scores.max():.3f}  "
      f"mean={valid_scores.mean():.3f}  std={valid_scores.std():.3f}")
print(f"  q05={np.quantile(valid_scores, 0.05):.3f}  "
      f"q50={np.quantile(valid_scores, 0.50):.3f}  "
      f"q95={np.quantile(valid_scores, 0.95):.3f}")

# 4) Top-K по убыванию энтропии. Сортируем индексы для быстрого Dataset.select().
top_k_local      = np.argsort(-valid_scores)[:SUBSAMPLE_SIZE]
selected_indices = np.sort(valid_indices[top_k_local])

top_scores = scores[selected_indices]
print(f"\nTop-{SUBSAMPLE_SIZE} по энтропии:")
print(f"  min={top_scores.min():.3f}  max={top_scores.max():.3f}  "
      f"mean={top_scores.mean():.3f}")

# 5) Освобождаем VRAM до загрузки обучающей копии модели в Cell 14.
del scorer_model, scorer_tok
gc.collect()
torch.cuda.empty_cache()
print("Скорер выгружен из VRAM.")
# ---- END STRATEGY BLOCK ----

assert len(selected_indices) == SUBSAMPLE_SIZE, (
    f"Стратегия должна вернуть ровно {SUBSAMPLE_SIZE} индексов, "
    f"получено {len(selected_indices)}"
)
assert len(set(selected_indices.tolist())) == SUBSAMPLE_SIZE, (
    "В selected_indices есть дубликаты — это сломает воспроизводимость"
)

# Сохраняем выбранные индексы — для отчёта и воспроизведения другими экспериментами.
np.save(INDICES_PATH, selected_indices)
print(f"Отобрано {len(selected_indices)} из {len(ds_full)} примеров")
print(f"Индексы сохранены в {INDICES_PATH}")


# ----- ШАГ 3: применение ChatML template + train/val split -----
# selected_indices живут в координатах pool_ds (после возможного POOL_LIMIT shuffle+select),
# не в координатах ds_full — поэтому select делаем именно из pool_ds.
ds = pool_ds.select(selected_indices.tolist())


def render(examples):
    texts = [
        tok.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples["conversations"]
    ]
    return {"text": texts}


ds = ds.map(render, batched=True, remove_columns=ds.column_names, desc="render chat template")

ds_split = ds.train_test_split(test_size=VAL_SIZE, seed=SEED)
print(f"\nTrain: {len(ds_split['train'])}, Val: {len(ds_split['test'])}")


# ----- Статистика длин (для предсказания packing-ratio и проверки на truncation) -----
sample_n = min(1000, len(ds_split["train"]))
lengths = [
    len(tok(ex["text"], add_special_tokens=False)["input_ids"])
    for ex in ds_split["train"].select(range(sample_n))
]
too_long = sum(1 for l in lengths if l > MAX_SEQ_LEN)
stats = {
    "avg_len_tokens": round(sum(lengths) / len(lengths), 1),
    "max_len_tokens": max(lengths),
    "pct_over_max_seq_len": round(too_long / len(lengths) * 100, 2),
    "expected_pack_ratio": round(MAX_SEQ_LEN / (sum(lengths) / len(lengths)), 2),
}
print("\nСтатистика длин (выборка 1000):")
print(json.dumps(stats, indent=2, ensure_ascii=False))
print(f"  → packing ожидает ~{stats['expected_pack_ratio']:.1f} примеров на одну упакованную последовательность")

print("\n--- Пример отрендеренного текста ---")
print(ds_split["train"][0]["text"][:500])

## Шаг 1.5 — Загрузка модели + LoRA

QLoRA: модель в 4-bit, LoRA-адаптер сверху. По плану:
- `r=16`, `lora_alpha=16`
- Целевые модули — все линейные слои attention и MLP
- `lora_dropout=0` обязательно при 4-bit
- `use_gradient_checkpointing="unsloth"` экономит ещё ~30% VRAM (Unsloth-овская реализация быстрее transformers-native)

После загрузки модели прогоняем `tokenizer` через `get_chat_template(..., chat_template="qwen-2.5")` — фиксируем тот же ChatML-шаблон Qwen2.5, который использовался в Cell 12 при скоринге и рендере `text`. Это важно для `train_on_responses_only`: маркеры `<|im_start|>user\n` и `<|im_start|>assistant\n` должны совпадать с тем, что реально лежит в input_ids. Это не потому, что base-модель не поддерживает `apply_chat_template`, а чтобы не зависеть от возможных изменений tokenizer config / версии `transformers` между подготовкой данных и запуском обучения.

**Чекпоинт:** доля обучаемых параметров — ожидаем 1–2%.


In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template


def load_model_with_lora():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
    )

    # Фиксируем Unsloth-овский ChatML-шаблон Qwen2.5 — гарантия согласованности
    # маркеров между prepare_data, тренировкой и инференсом
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Обучаемых параметров: {trainable:,} ({100 * trainable / total:.2f}%)")

    return model, tokenizer


model, tokenizer = load_model_with_lora()

## Шаг 2.1 — Обучение с `packing=True` + `train_on_responses_only`

Этот стек **зафиксирован** на обновлённом baseline-этапе и должен быть одинаков для всех экспериментов с data selection. Менять параметры здесь — значит ломать сравнимость с `baseline_masked_packed.ipynb` и другими стратегиями.

- 1 эпоха на ~85 500 train-примерах из отобранных 90k, **эффективный batch = 8** (`BATCH_SIZE=4 × GRAD_ACCUM=2`).
- `MAX_SEQ_LEN = 2048`. На `d0rj/ru-instruct` обрезается ~2–3% длинных примеров.
- **`packing=True`** ставится атрибутом на `training_args` (Unsloth compiled wrapper не пропускает его в kwargs `SFTConfig`). Новый режим `90k × 1` даёт примерно столько же optimizer steps, сколько старый `30k × 3`, но без трёх повторных проходов по одним и тем же данным.
- **`train_on_responses_only`** вызывается **после** `SFTTrainer(...)`. Маскирует prompt-токены под `-100`. Корректно обрабатывает несколько пар маркеров в одной упакованной последовательности (Unsloth discussion #2828).
- `warmup_ratio=0.03` вместо фиксированных `warmup_steps=500`: warmup масштабируется от реального числа шагов.
- **Не дублируем** `gradient_checkpointing=True` в `SFTConfig` — Unsloth уже включил его в `get_peft_model(use_gradient_checkpointing="unsloth")`.
- **Не используем** `assistant_only_loss=True` — несовместим с pre-rendered text (см. подробнее в Cell 0).


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

precision = detect_precision()
print("Используем bf16")

eval_subset = ds_split["test"].select(range(min(EVAL_SAMPLES, len(ds_split["test"]))))
print(f"Train: {len(ds_split['train'])}, in-loop eval: {len(eval_subset)}")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    # gradient_checkpointing уже включён через use_gradient_checkpointing="unsloth"
    # в FastLanguageModel.get_peft_model().
    optim="adamw_8bit",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,

    bf16=precision["bf16"],
    fp16=precision["fp16"],

    max_length=MAX_SEQ_LEN,

    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",

    dataloader_num_workers=0,
    dataloader_pin_memory=True,

    seed=SEED,
    data_seed=SEED,
)

# packing ставим атрибутом на объекте, а не kwargs в SFTConfig:
# Unsloth-овский UnslothSFTTrainer.py (compiled cache) пересобирает SFTConfig
# с явным белым списком аргументов. На group_by_length мы уже натыкались на
# TypeError. Trainer читает self.args.packing в момент _prepare_dataset,
# поэтому setattr ДО создания SFTTrainer срабатывает гарантированно.
training_args.packing = True
training_args.eval_packing = False   # eval per-пример: eval_loss сравним по сэмплам с baseline_masked

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=ds_split["train"],
    eval_dataset=eval_subset,
)

# Маскируем prompt-токены: labels = -100 для всего, кроме ответа ассистента.
# С packing в одной упакованной последовательности будет несколько пар маркеров
# user/assistant — train_on_responses_only в свежих версиях Unsloth (см.
# discussion #2828) корректно обрабатывает все пары.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

### Sanity check: packing + маскирование

Проверяем три вещи:
1. **Packing работает**: `seq_length` в батче должна быть близка к `MAX_SEQ_LEN`, а **не** ~600 (средняя длина одного примера). Если seq_length маленький — packing не включился.
2. **Несколько assistant-сегментов в одной упакованной последовательности**: считаем число «контигуальных групп» токенов с `label != -100`. Если packing работает + маска работает, ожидаем 3–5 сегментов в каждой packed-строке.
3. **Trained-токены — это и правда ответы ассистента**: декодим их и читаем глазами. Trained-текст — конкатенация всех ответов в упакованной последовательности, разделённых `<|im_end|>`.

**Когда стоит начать переживать:**
- `seq_length` в батче ≈ 200–600 → **packing не сработал**, проверь `training_args.packing = True` и версию Unsloth (нужна свежая, см. PR #3566).
- `aggregate_ratio < 1%` или `> 95%` — `train_on_responses_only` не пробил маркеры. Скорее всего, расхождение между ChatML-шаблоном в Cell 12 и маркерами в Cell 16 (`<|im_start|>user\n`). В этом случае посмотри декодинг начала первой последовательности — что именно там стоит вместо ожидаемого маркера.
- `assistant segments per packed sequence == 1` — несмотря на packing, маркер ищется только по первой паре. Нужна более новая версия Unsloth.
- В декодинге trained-токенов попадаются куски user-промптов или русский от лица user-а — chat template неправильный.


In [ ]:
batch = next(iter(trainer.get_train_dataloader()))
input_ids_batch = batch["input_ids"]
labels_batch    = batch["labels"]
B, L = input_ids_batch.shape
print(f"Batch: {B} packed-последовательностей × {L} токенов")
print(f"padding_side: {tokenizer.padding_side}")

# 1) Packing работает? seq_length должна быть близка к MAX_SEQ_LEN.
print(f"\n--- (1) Packing check ---")
print(f"L = {L}, MAX_SEQ_LEN = {MAX_SEQ_LEN}")
if L < MAX_SEQ_LEN * 0.7:
    print(f"WARNING: L = {L} сильно меньше MAX_SEQ_LEN — packing, похоже, НЕ сработал.")
    print("        Проверь training_args.packing = True и что Unsloth поддерживает packing.")
else:
    print(f"OK: L близок к MAX_SEQ_LEN — packing работает.")

# 2) Сколько assistant-сегментов в каждой последовательности.
#    Считаем как число «контигуальных групп токенов с label != -100».
print(f"\n--- (2) Assistant segments per packed sequence ---")
for i in range(B):
    labels = labels_batch[i]
    mask   = labels != -100
    # Число «переходов» 0→1 в маске = число assistant-сегментов.
    transitions = ((mask[1:].int() - mask[:-1].int()) == 1).sum().item()
    if mask[0]:
        transitions += 1
    print(f"  [{i}] assistant segments: {transitions}")

# 3) Per-sequence статистика + декодинг.
print(f"\n--- (3) Per-sequence ratio + decoded trained tokens ---")
for i in range(B):
    input_ids = input_ids_batch[i]
    labels    = labels_batch[i]
    mask      = labels != -100
    n_masked  = (~mask).sum().item()
    n_trained = mask.sum().item()
    ratio_i   = n_trained / L

    trained_text = tokenizer.decode(input_ids[mask], skip_special_tokens=False)
    print(f"  [{i}] masked={n_masked:5d}  trained={n_trained:5d}  ratio={ratio_i:6.2%}")
    print(f"      trained → {trained_text[:160]!r}")

# 4) Агрегированный ratio по всему батчу.
total_trained = (labels_batch != -100).sum().item()
total_tokens  = labels_batch.numel()
agg_ratio     = total_trained / total_tokens
print(f"\n--- (4) Aggregate ---")
print(f"trained_tokens = {total_trained}/{total_tokens} → ratio = {agg_ratio:.2%}")

if agg_ratio < 0.01:
    print("WARNING: маска НЕ сработала — train_on_responses_only не пробил маркер.")
    print("Декодинг начала первой последовательности (для отладки):")
    print(repr(tokenizer.decode(input_ids_batch[0][:120])))
    print("Сравни с ожидаемым '<|im_start|>user\\n...'")
elif agg_ratio > 0.95:
    print("WARNING: маска НЕ сработала — почти всё попадает в loss.")
    print("Маркер instruction_part='<|im_start|>user\\n' видимо не нашёлся.")
else:
    print("OK: маскирование работает корректно.")

In [ ]:
import os

ckpt_dirs = []
if os.path.isdir(OUTPUT_DIR):
    ckpt_dirs = sorted(d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-"))

if ckpt_dirs:
    print(f"Найдено {len(ckpt_dirs)} чекпоинт(ов): {ckpt_dirs}")
    print(f"Продолжаем с последнего: {ckpt_dirs[-1]}")
    trainer.train(resume_from_checkpoint=True)
else:
    print("Чекпоинтов нет — обучение с нуля.")
    trainer.train()

In [ ]:
with open(LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, indent=2, ensure_ascii=False)

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Адаптер: {ADAPTER_DIR}")
print(f"Логи:    {LOG_PATH}")
print(f"Шагов сделано: {trainer.state.global_step}")

## Шаг 2.2 — Оценка качества (perplexity)

Загружаем **сохранённый адаптер**, а не свежую модель с нулевыми LoRA-весами.

Загрузку делаем через `FastLanguageModel.from_pretrained(ADAPTER_DIR, ...)`: Unsloth увидит `adapter_config.json`, сам подтянет базовую модель в 4-bit и навесит LoRA. Использовать «голый» `AutoModelForCausalLM.from_pretrained` нельзя — после `import unsloth` (ячейка 6) в `transformers` глобально пропатчен `Qwen2Attention.forward`, который ждёт атрибут `apply_qkv`.

`FastLanguageModel.for_inference(model)` дополнительно ускоряет инференс (отключает gradient checkpointing, переключает в eval).

Метрика — **perplexity по полному тексту примера**. В этом ноутбуке `text` уже отрендерен на этапе подготовки данных (Cell 12) через явно зафиксированный Qwen2.5 ChatML-шаблон. Для сравнимости другие стратегии должны использовать тот же рендер и тот же common val.

> Перед загрузкой освобождаем VRAM от обучающей копии модели/тренера через `del trainer, model` + `torch.cuda.empty_cache()`.


In [ ]:
import json
import math, time, gc

# Освобождаем VRAM от обучающей копии модели/тренера
del trainer, model
gc.collect()
torch.cuda.empty_cache()


def load_trained_model():
    model, tok = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_DIR,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    return model, tok


def compute_perplexity(model, tok, texts):
    losses = []
    for text in texts:
        inputs = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to(model.device)
        with torch.no_grad():
            loss = model(**inputs, labels=inputs["input_ids"]).loss
        losses.append(loss.item())
    return math.exp(sum(losses) / len(losses))


eval_model, eval_tok = load_trained_model()

n_eval = min(FINAL_EVAL_SAMPLES, len(ds_split["test"]))
# text уже отрендерен в Cell 12 через явно зафиксированный Qwen2.5 ChatML-шаблон.
texts = [ex["text"] for ex in ds_split["test"].select(range(n_eval))]

t0 = time.time()
ppl = compute_perplexity(eval_model, eval_tok, texts)
elapsed = time.time() - t0

metrics = {
    "strategy":          STRATEGY,
    "perplexity":        round(ppl, 4),
    "eval_time_sec":     round(elapsed, 2),
    "samples_evaluated": n_eval,
    "adapter":           ADAPTER_DIR,
    "model":             MODEL_NAME,
    "subsample_size":    SUBSAMPLE_SIZE,
    "training_config":   f"packing=True, train_on_responses_only, subsample={SUBSAMPLE_SIZE}, epochs={NUM_EPOCHS}, warmup_ratio={WARMUP_RATIO}",
    # NB: это full-text PPL. Verdict ниже строится по assistant-only PPL на common val.
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"Perplexity:    {ppl:.4f}")
print(f"Время:         {elapsed:.1f} сек на {n_eval} примерах")
print(f"Метрики:       {METRICS_PATH}")

## Шаг 2.3 — Сравнение с `baseline_masked_packed_base_90k_1ep_noleak` на `assistant_only_PPL`

**Зачем нужна отдельная метрика.** Full-text PPL (Cell 22) — методологически неконсистентная метрика для masked-моделей (training distribution mismatch с prompt-токенами). `assistant_only_PPL` считает CE-loss только на тех токенах, на которых модель училась через `train_on_responses_only` — корректно для всех стратегий из этого workflow.

**КРИТИЧЕСКИ ВАЖНО — общий val.** Каждая стратегия отбора создаёт свой `ds_split["test"]` из своих же отобранных данных. Распределения собственных val у разных стратегий разные, поэтому PPL на собственном val между стратегиями **несравнима**.

Решение: для сравнения с baseline используем **общий внешний val-сет**: `shuffle(seed=SEED)[0:COMMON_VAL_HOLDOUT_SIZE]`. Все train/selection/scoring pools начинаются после этого holdout, поэтому common val не попадает в обучение ни у baseline, ни у selection-стратегий.

PPL на собственном val тоже считаем — как диагностику обучения (пишется как `perplexity_assistant_only_on_own_val`), но **не** используется для verdict. PPL на общем val записывается как `perplexity_assistant_only_on_common_val` и идёт в `comparison_to_baseline`.

Источник числа baseline: `outputs/baseline_masked_packed_base_90k_1ep_noleak/final_metrics.json`. Старый fallback `2.4891` удалён: он относится к Instruct + `30k × 3` и после обновления протокола невалиден.

**Verdict с порогом шума 0.03** — временный практический порог из старого baseline-этапа. Если появятся повторные прогоны нового baseline, лучше заменить его на эмпирическую дисперсию нового протокола.


In [ ]:
import math, json, torch
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer


def compute_assistant_only_perplexity(model, tok, examples, max_length=MAX_SEQ_LEN):
    """PPL только на assistant-токенах. Принимает примеры в любом из форматов:
    - 'text' (предотрендеренный chat template — как в этом ноутбуке);
    - 'messages' / 'conversations' (raw conversational).
    Маркеры — ChatML-формат Qwen2.5 (<|im_start|>user\n / <|im_start|>assistant\n).
    """
    user_marker_ids = tok("<|im_start|>user\n",      add_special_tokens=False)["input_ids"]
    asst_marker_ids = tok("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]

    total_loss   = 0.0
    total_tokens = 0
    skipped      = 0

    for ex in examples:
        if "text" in ex:
            text = ex["text"]
        elif "messages" in ex:
            text = tok.apply_chat_template(
                ex["messages"], tokenize=False, add_generation_prompt=False,
            )
        elif "conversations" in ex:
            text = tok.apply_chat_template(
                ex["conversations"], tokenize=False, add_generation_prompt=False,
            )
        else:
            raise ValueError(f"Жду 'text', 'messages' или 'conversations', есть: {list(ex.keys())}")

        inputs = tok(text, return_tensors="pt", truncation=True,
                     max_length=max_length).to(model.device)
        input_ids = inputs["input_ids"][0]

        labels = input_ids.clone()
        labels[:] = -100  # маскируем всё...
        ids_list = input_ids.tolist()
        i = 0
        while i < len(ids_list):
            if ids_list[i:i + len(asst_marker_ids)] == asst_marker_ids:
                start = i + len(asst_marker_ids)
                end = len(ids_list)
                for j in range(start, len(ids_list) - len(user_marker_ids) + 1):
                    if ids_list[j:j + len(user_marker_ids)] == user_marker_ids:
                        end = j
                        break
                labels[start:end] = input_ids[start:end]   # ...и размаскируем assistant-сегменты
                i = end
            else:
                i += 1

        n_assistant = (labels != -100).sum().item()
        if n_assistant == 0:
            skipped += 1
            continue

        with torch.no_grad():
            out = model(input_ids=input_ids.unsqueeze(0), labels=labels.unsqueeze(0))

        total_loss   += out.loss.item() * n_assistant
        total_tokens += n_assistant

    if total_tokens == 0:
        raise RuntimeError("Не нашлось ни одного assistant-токена. Проверь маркеры/chat template.")
    if skipped:
        print(f"  WARNING: пропущено {skipped} примеров без assistant-сегментов (truncation?)")
    print(f"  Усреднено по {total_tokens} assistant-токенам, {len(examples) - skipped} примеров")
    return math.exp(total_loss / total_tokens)


# --- (1) PPL на СОБСТВЕННОМ val — диагностика, НЕ для сравнения между стратегиями.
n_eval = min(FINAL_EVAL_SAMPLES, len(ds_split["test"]))
own_val = ds_split["test"].select(range(n_eval))
print(f"--- PPL на СОБСТВЕННОМ val ({len(own_val)} примеров, диагностика) ---")
ppl_asst_own = compute_assistant_only_perplexity(eval_model, eval_tok, own_val)
print(f"  → {ppl_asst_own:.4f}")


# --- (2) PPL на ОБЩЕМ val — единственная метрика, по которой стратегии сравнимы.
print(
    f"\nПостроение ОБЩЕГО val (reserved {COMMON_VAL_HOLDOUT_SIZE:,}, seed=SEED; "
    "исключён из train/selection pools)..."
)
stock_tok = get_chat_template(
    AutoTokenizer.from_pretrained(MODEL_NAME),
    chat_template="qwen-2.5",
)
ds_full = load_dataset(DATASET_NAME, split="train")
ds_common = ds_full.shuffle(seed=SEED).select(range(COMMON_VAL_HOLDOUT_SIZE))


def _render_common(examples):
    texts = [
        stock_tok.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples["conversations"]
    ]
    return {"text": texts}


ds_common = ds_common.map(
    _render_common, batched=True,
    remove_columns=ds_common.column_names,
    desc="render common val",
)
common_val_size = min(FINAL_EVAL_SAMPLES, len(ds_common))
common_val = ds_common.select(range(common_val_size))

print(f"\n--- PPL на ОБЩЕМ val ({len(common_val)} примеров, главная метрика) ---")
ppl_asst_common = compute_assistant_only_perplexity(eval_model, eval_tok, common_val)
print(f"  → {ppl_asst_common:.4f}")


# --- (3) Сравнение с baseline на ОБЩЕМ val.
BASELINE_METRICS_PATH = f"{OUTPUT_BASE}/outputs/baseline_masked_packed_base_90k_1ep_noleak/final_metrics.json"
NOISE_FLOOR           = 0.03

try:
    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"Не найден baseline metrics: {BASELINE_METRICS_PATH}. "
        "Сначала прогони обновлённый baseline_masked_packed.ipynb."
    ) from exc

if "perplexity_assistant_only" not in baseline:
    raise KeyError(f"В {BASELINE_METRICS_PATH} нет ключа 'perplexity_assistant_only'.")

baseline_ppl_asst = baseline["perplexity_assistant_only"]
baseline_source = BASELINE_METRICS_PATH

delta_abs = ppl_asst_common - baseline_ppl_asst
delta_pct = (ppl_asst_common / baseline_ppl_asst - 1) * 100

if delta_abs < -NOISE_FLOOR:
    verdict = f"{STRATEGY} ЛУЧШЕ baseline_masked_packed_base_90k_1ep_noleak (статзначимо)"
elif abs(delta_abs) <= NOISE_FLOOR:
    verdict = f"статистически неотличимо от baseline (|Δ| ≤ {NOISE_FLOOR} — шум)"
else:
    verdict = f"{STRATEGY} ХУЖЕ baseline_masked_packed_base_90k_1ep_noleak (статзначимо)"


# --- (4) Запись итоговых метрик.
with open(METRICS_PATH) as f:
    metrics = json.load(f)
metrics["perplexity_assistant_only_on_own_val"]    = round(ppl_asst_own, 4)
metrics["perplexity_assistant_only_on_common_val"] = round(ppl_asst_common, 4)
metrics["comparison_to_baseline"] = {
    "val_set":            f"reserved common val (shuffle(seed=SEED)[0:{COMMON_VAL_HOLDOUT_SIZE}], first {len(common_val)} examples)",
    "baseline_strategy":  "random_90000 (baseline_masked_packed_base_90k_1ep_noleak)",
    "baseline_ppl_asst":  round(baseline_ppl_asst, 4),
    "baseline_source":    baseline_source,
    "strategy_ppl_asst":  round(ppl_asst_common, 4),
    "delta_absolute":     round(delta_abs, 4),
    "delta_percent":      round(delta_pct, 2),
    "verdict":            verdict,
    "noise_floor":        NOISE_FLOOR,
    "samples_evaluated":  len(common_val),
    "note": (
        "PPL on OWN val is for diagnostics only — strategies have differently-"
        "distributed own vals, so own-val PPL is incomparable between strategies. "
        "The verdict uses COMMON val reserved before any train/selection/scoring pool."
    ),
}
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 78)
print(f"  Сравнение {STRATEGY} с baseline_masked_packed_base_90k_1ep_noleak на ОБЩЕМ val")
print("=" * 78)
print(f"  baseline:               ppl_asst = {baseline_ppl_asst:.4f}")
print(f"  {STRATEGY[:26]:26s}  ppl_asst = {ppl_asst_common:.4f}  (на общем val)")
print(f"                                         {ppl_asst_own:.4f}  (на собственном val, diag)")
print(f"  Δ:                      {delta_abs:+.4f}  ({delta_pct:+.2f}%)")
print(f"  noise:                  ±{NOISE_FLOOR}")
print(f"  verdict:                {verdict}")
print("=" * 78)
print(f"\nМетрики обновлены: {METRICS_PATH}")


## (Опционально) Сохранение результатов между сессиями

Сессия Colab умрёт через несколько часов — всё в `/content/` пропадёт. Рекомендуемая схема:

| Что | Куда |
|---|---|
| LoRA-адаптер (~30 МБ) | HuggingFace Hub (приватный репо) |
| `training_log.json`, `final_metrics.json` | Скачать локально и закоммитить в git |

Альтернатива — смонтировать Google Drive и скопировать туда `outputs/{STRATEGY}/` целиком. Если у тебя нет HF-аккаунта или не хочется заводить токен — следующие две ячейки можно пропустить.

Токен берём через `getpass`, чтобы он не попал в git.

In [ ]:
# Раскомментируй и впиши свой username + желаемое имя репо.
# По умолчанию имя репо собирается из STRATEGY — заменишь только username.
# HF_REPO_ID = f"your-username/qwen2.5-1.5b-base-ru-{STRATEGY}"
HF_REPO_ID = None

if HF_REPO_ID is not None:
    from huggingface_hub import HfApi, login
    from getpass import getpass

    login(token=getpass("HF token (write scope): "))

    api = HfApi()
    api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=ADAPTER_DIR,
        repo_id=HF_REPO_ID,
        commit_message=f"{STRATEGY} adapter ({SUBSAMPLE_SIZE} subsample, {NUM_EPOCHS} epochs, packing=True, train_on_responses_only)",
    )
    api.upload_file(
        path_or_fileobj=METRICS_PATH,
        path_in_repo="final_metrics.json",
        repo_id=HF_REPO_ID,
    )
    api.upload_file(
        path_or_fileobj=LOG_PATH,
        path_in_repo="training_log.json",
        repo_id=HF_REPO_ID,
    )
    print(f"Адаптер и логи запушены в https://huggingface.co/{HF_REPO_ID}")
else:
    print("HF_REPO_ID не задан — пропускаем пуш на Hub.")
    print("Перед закрытием Colab скачай:")
    print(f"  - {ADAPTER_DIR}/")
    print(f"  - {LOG_PATH}")
    print(f"  - {METRICS_PATH}")